# Sri Lanka Flood Prediction Model Training
This notebook details the Exploratory Data Analysis (EDA) and Hyperparameter Optimization for predicting flood risk levels and estimating flood depth in Sri Lanka.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_squared_error, mean_absolute_error
import pickle
import os

sns.set_theme(style='whitegrid')

## 1. Load the Actual Dataset

In [ ]:
dataset_path = r'D:\flood_predict\backend\dataset\sri_lanka_flood_risk_dataset.csv'
df = pd.read_csv(dataset_path)
df.head()

## 2. Basic Dataset Info

In [ ]:
print('Dataset Shape:', df.shape)
df.info()
df.describe()

## 3. Exploratory Data Analysis (EDA)
Let's visualize target distributions, feature correlations, and hydrological relations.

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='flood_occurred', palette='Set2')
plt.title('Distribution of Flood Events (0: No Flood, 1: Flood)')
plt.show()

### Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 8))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.show()

### Hydrological Relationships: Daily Rainfall vs River Level

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='daily_rainfall_mm', y='river_level_m', hue='flood_occurred', palette='Set1', alpha=0.7)
plt.title('Rainfall vs River Level by Flood Occurrence')
plt.show()

## 4. Feature Mapping & Target Engineering

In [ ]:
def map_risk_level(depth):
    if depth == 0.0:
        return 0
    elif depth <= 1.0:
        return 1
    elif depth <= 2.5:
        return 2
    else:
        return 3

df['risk_level'] = df['flood_depth_m'].apply(map_risk_level)

plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='risk_level', palette='viridis')
plt.title('Engineered Risk Level Distribution (0:Low, 1:Med, 2:High, 3:Crit)')
plt.show()

## 5. Model Training & Hyperparameter Tuning

In [ ]:
feature_cols = ['daily_rainfall_mm', '3_day_cumulative_rain', 'rate_of_rise', 'elevation_m', 'slope_degrees', 'distance_to_river_km']
X = df[feature_cols]
y_class = df['risk_level']
y_depth = df['flood_depth_m']

X_train, X_test, y_c_train, y_c_test = train_test_split(X, y_class, test_size=0.2, random_state=42, stratify=y_class)
_, _, y_d_train, y_d_test = train_test_split(X, y_depth, test_size=0.2, random_state=42)

### Train and Optimize Random Forest Classifier

In [ ]:
# Define grid search parameter space
rf_clf = RandomForestClassifier(random_state=42)
clf_param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [6, 8, 10],
    'min_samples_split': [2, 5]
}
clf_grid = GridSearchCV(rf_clf, clf_param_grid, cv=3, scoring='accuracy', n_jobs=-1)
clf_grid.fit(X_train, y_c_train)
best_clf = clf_grid.best_estimator_
print('Best Classifier parameters found:', clf_grid.best_params_)

### Evaluate Classifier

In [ ]:
y_c_pred = best_clf.predict(X_test)
print('Test Accuracy:', accuracy_score(y_c_test, y_c_pred))
print(classification_report(y_c_test, y_c_pred, target_names=['Low', 'Medium', 'High', 'Critical']))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_c_test, y_c_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Low', 'Med', 'High', 'Crit'], yticklabels=['Low', 'Med', 'High', 'Crit'])
plt.title('Classification Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

### Train and Optimize Random Forest Regressor

In [ ]:
rf_reg = RandomForestRegressor(random_state=42)
reg_param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [6, 8, 10],
    'min_samples_split': [2, 5]
}
reg_grid = GridSearchCV(rf_reg, reg_param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
reg_grid.fit(X_train, y_d_train)
best_reg = reg_grid.best_estimator_
print('Best Regressor parameters found:', reg_grid.best_params_)

### Evaluate Regressor

In [ ]:
y_d_pred = best_reg.predict(X_test)
mse = mean_squared_error(y_d_test, y_d_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_d_test, y_d_pred)
print(f'Test MSE: {mse:.4f}')
print(f'Test RMSE: {rmse:.4f} meters')
print(f'Test MAE: {mae:.4f} meters')